# 03 — LUNA16 classical ML (SVM / RF / KNN)

5-fold stratified outer CV with inner 3-fold grid search per model. Both default-threshold and Youden-calibrated metrics are saved per fold to `results/luna16_<model>.json`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import yaml

from utils.seed import set_seed
from utils.models import CLASSICAL_REGISTRY
from utils.training import train_classical_cv, save_fold_results
from utils.metrics import aggregate_folds
from utils.stats import format_results_table

with open('../configs/luna16.yaml') as f:
    cfg = yaml.safe_load(f)
set_seed(cfg['seed'])

cache_dir = Path('..') / cfg['data']['cache_dir']
data = np.load(cache_dir / 'features.npz', allow_pickle=True)
X, y = data['X'], data['y']
groups = data['seriesuid']  # GroupKFold by CT scan to avoid patch-level leakage
print('X', X.shape, 'pos rate:', y.mean())

X (1070, 91) pos rate: 0.20093457943925233


In [2]:
results_dir = Path('..') / cfg['paths']['results']
results_dir.mkdir(parents=True, exist_ok=True)

aggregated = {}
for name, (builder, grid) in CLASSICAL_REGISTRY.items():
    print(f'\n=== {name} ===')
    folds = train_classical_cv(
        X, y, builder, grid,
        n_splits=cfg['cv']['n_splits'],
        inner_splits=cfg['cv']['inner_splits'],
        scoring=cfg['cv']['scoring'],
        seed=cfg['seed'],
        groups=groups,
        verbose=True,
    )
    save_fold_results(folds, results_dir / f'luna16_{name}.json')
    aggregated[name] = aggregate_folds([f.metrics_calibrated for f in folds])


=== SVM ===
[fold 0] train=855 test=215
[fold 0] best_params={'C': 10, 'gamma': 0.1, 'kernel': 'rbf'} thr=0.377 AUC=0.704 BalAcc(cal)=0.557
[fold 1] train=856 test=214
[fold 1] best_params={'C': 1, 'gamma': 0.1, 'kernel': 'rbf'} thr=0.170 AUC=0.782 BalAcc(cal)=0.683
[fold 2] train=856 test=214
[fold 2] best_params={'C': 100, 'gamma': 0.01, 'kernel': 'rbf'} thr=0.238 AUC=0.731 BalAcc(cal)=0.648
[fold 3] train=856 test=214
[fold 3] best_params={'C': 1, 'gamma': 'scale', 'kernel': 'rbf'} thr=0.280 AUC=0.722 BalAcc(cal)=0.659
[fold 4] train=857 test=213
[fold 4] best_params={'C': 1, 'gamma': 0.1, 'kernel': 'rbf'} thr=0.234 AUC=0.755 BalAcc(cal)=0.739

=== RF ===
[fold 0] train=855 test=215
[fold 0] best_params={'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200} thr=0.471 AUC=0.710 BalAcc(cal)=0.557
[fold 1] train=856 test=214
[fold 1] best_params={'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100} thr=0.485 AUC=0.762 BalAcc(cal)=0.689
[fold 2] train=856 test=214
[fo

In [3]:
table = format_results_table(aggregated)
table

,accuracy,sensitivity,specificity,precision,f1,balanced_accuracy,auc_roc,pr_auc
model,,,,,,,,
SVM,0.704 ± 0.052,0.577 ± 0.166,0.737 ± 0.082,0.356 ± 0.066,0.433 ± 0.089,0.657 ± 0.066,0.739 ± 0.030,0.423 ± 0.081
RF,0.751 ± 0.036,0.403 ± 0.123,0.837 ± 0.046,0.388 ± 0.076,0.389 ± 0.084,0.620 ± 0.054,0.741 ± 0.023,0.406 ± 0.060
KNN,0.777 ± 0.048,0.156 ± 0.349,0.935 ± 0.146,0.073 ± 0.163,0.099 ± 0.222,0.545 ± 0.102,0.727 ± 0.029,0.392 ± 0.041


In [4]:
import json

for name, (_, grid) in CLASSICAL_REGISTRY.items():
    folds_path = results_dir / f'luna16_{name}.json'
    with folds_path.open() as f:
        raw = json.load(f)
    print(name, 'selected hparams per fold:')
    for r in raw:
        print(' ', r['fold'], r['best_params'])

SVM selected hparams per fold:
  0 {'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}
  1 {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
  2 {'C': 100, 'gamma': 0.01, 'kernel': 'rbf'}
  3 {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
  4 {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
RF selected hparams per fold:
  0 {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}
  1 {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}
  2 {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 500}
  3 {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 200}
  4 {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 200}
KNN selected hparams per fold:
  0 {'n_neighbors': 21, 'p': 2, 'weights': 'distance'}
  1 {'n_neighbors': 21, 'p': 2, 'weights': 'distance'}
  2 {'n_neighbors': 21, 'p': 2, 'weights': 'distance'}
  3 {'n_neighbors': 15, 'p': 2, 'weights': 'distance'}
  4 {'n_neighbors': 21, 'p': 2, 'weights': 'uniform'}
